# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
# 在此写下你的答案：
# 1.代表每个客户的所有信息
# 2.目标变量是Churn这一列
# 3.CustomerID 是身份，而非属性，它的数值大小没有数学意义。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [ ]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    missing_count = data.isnull().sum()
    missing_rate = (data.isnull().mean()*100).round(2)

    report = pd.DataFrame({
        '数据类型':data.dtypes,
        '缺失数量':missing_count,
        '缺失比例(%)':missing_rate,
        '唯一值数量':data.nunique(),
    })

    report.reset_index(inplace=True)
    report.rename(columns={'index':'字段名'}, inplace=True)

    return report


# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
quality_before

### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [ ]:
# TODO：完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df['CustomerID'].value_counts().max() - 1)
print(raw_df["Churn"].value_counts())
print("流失率：",((raw_df['Churn'] == 0).sum()/raw_df['Churn'].sum()).round(2),'%' )

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [ ]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [ ]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    """
    # 1. 复制数据
    cleaned_df = data.copy()
    logs = []

    # 2. 删除完全重复行
    dup_count = cleaned_df.duplicated().sum()
    if dup_count > 0:
        cleaned_df.drop_duplicates(inplace=True)
        logs.append({'步骤': '删除完全重复行', '详情': '删除所有字段完全相同的行', '影响数量': dup_count})
    else:
        logs.append({'步骤': '删除完全重复行', '详情': '无重复数据', '影响数量': 0})

    # 3. 缺失值填补
    for col in NUMERIC_MISSING_COLS:
        if col not in cleaned_df.columns:
            continue
        missing_num = cleaned_df[col].isnull().sum()
        if missing_num:
            median_val = cleaned_df[col].median()
            cleaned_df[col] = cleaned_df[col].fillna(median_val)
            logs.append({
                '步骤': '缺失值填补',
                '详情': f"列 '{col}' 使用中位数填补",
                '影响数量': missing_num
            })
        else:
            logs.append({
                '步骤': '缺失值填补',
                '详情': f"列 '{col}' 无缺失",
                '影响数量': 0
            })

     # 4. 类别标准化，避免链式赋值
    for col, mapping_dict in CATEGORY_MAPPINGS.items():
        if col not in cleaned_df.columns:
            continue
        for old_val, new_val in mapping_dict.items():
            count = (cleaned_df[col] == old_val).sum()
            if not count:
                continue
            cleaned_df[col] = cleaned_df[col].replace(old_val, new_val)
            logs.append({
                '步骤': '类别标准化',
                '详情': f"列 {col}: '{old_val}' -> '{new_val}'",
                '影响数量': count
            })


    # 5. 类型转换
    for col in ['Churn', 'Complain']:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(int)

    # 6. 生成日志并返回
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log


### 任务 3：运行清洗函数并查看日志

In [ ]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

print(cleaning_log)
print(cleaned_df.head())

---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [ ]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
# 设定分箱区间：0-12月(新用户), 12-24月(稳定), 24-36月(老用户), 36月+(忠诚)
tenure_bins = [0, 12, 24, 36, float('inf')]
tenure_labels = ['新用户', '稳定用户', '老用户', '忠诚用户']

# 使用 pd.cut 进行分箱，right=False 表示左闭右开区间 [a, b)
cleaned_df['TenureGroup'] = pd.cut(
    cleaned_df['Tenure'],
    bins=tenure_bins,
    labels=tenure_labels,
    right=False
)

# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df['IsMobileLogin'] = (cleaned_df['PreferredLoginDevice'] == 'Mobile Phone').astype(int)

# TODO：生成 outlier_report（每行对应一个待检查字段）
# 选定需要检查异常值的数值字段
cols_to_check = [
    'Tenure', 'WarehouseToHome', 'HourSpendOnApp',
    'OrderAmountHikeFromlastYear', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder'
]

outlier_results = []
for col in cols_to_check:
    if col in cleaned_df.columns:
        # 调用刚才定义的函数获取摘要字典
        summary = iqr_outlier_summary(cleaned_df[col])
        # 将字段名加入字典，便于后续识别
        summary['字段名'] = col
        outlier_results.append(summary)

# 将列表转换为 DataFrame
outlier_report = pd.DataFrame(outlier_results)

# 调整列顺序，让字段名在第一列
outlier_report = outlier_report[['字段名', 'Q1', 'Q3', '下限', '上限', '候选异常值数量']]

outlier_report

### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [ ]:
# TODO：完成业务规则检查
rules_check = [
    ("使用时长小于 0",     lambda df: (df['Tenure'] < 0).sum()),
    ("仓库距离小于 0",     lambda df: (df['WarehouseToHome'] < 0).sum()),
    ("订单数小于或等于 0", lambda df: (df['OrderCount'] <= 0).sum()),
    # 注意：通常数据集中对应列为 CashbackAmount，若不存在则记为0
    ("返现金额小于 0",     lambda df: (df['CashbackAmount'] < 0).sum() if 'CashbackAmount' in df.columns else 0)
]

# 收集结果
report_data = []
for rule_name, check_func in rules_check:
    try:
        count = check_func(cleaned_df)
    except KeyError:
        count = 0 # 若列不存在，视为0
    report_data.append({"规则": rule_name, "不合规记录数": count})

business_rule_report = pd.DataFrame(report_data)
business_rule_report

# 处理结论：

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [ ]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# TODO：输出 outlier_report 和 business_rule_report
print("\n--- 异常值检测报告 ---")
print(outlier_report)

print("\n--- 业务规则检查报告 ---")
print(business_rule_report)

# TODO：输出交付文件的路径
print("\n--- 交付文件路径清单 ---")
file_paths = [
    OUTPUT_DIR / "data_quality_before.csv",
    OUTPUT_DIR / "data_quality_after.csv",
    OUTPUT_DIR / "cleaning_log.csv",
    OUTPUT_DIR / "ecommerce_customer_cleaned.csv"
]
for path in file_paths:
    print(f" {path}")

## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？